In [1]:
import zipfile
import os
import random
import gc
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

extract_path = "."
zip_filename = "comp34612.zip"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_path)

from engine import Engine
from gui import GUI

import numpy as np


class Leader:
    _subclass_registry = {}

    def __init__(self, name, engine):
        self.name = name
        self.engine = engine

    @classmethod
    def cleanup_old_subclasses(cls):
        """
        A function to remove old subclasses before defining new ones.
        """
        existing_subclasses = list(cls.__subclasses__())

        for subclass in existing_subclasses:
            subclass_name = subclass.__name__
            if subclass_name in cls._subclass_registry:
                del cls._subclass_registry[subclass_name]
                del subclass
        gc.collect()

    @classmethod
    def update_subclass_registry(cls):
        """
        A function to update registry after cleaning up old subclasses.
        """
        cls.cleanup_old_subclasses()
        cls._subclass_registry = {subclass.__name__: subclass for subclass in cls.__subclasses__()}

    def new_price(self, date):
        """
        A function for setting the new price of each day.
        :param date: date of the day to be updated
        :return: (float) price for the day
        """
        pass

    def get_price_from_date(self, date):
        """
        A function for getting the price set on a date.
        :param date: (int) date to get the price from
        :return: a tuple (leader_price, follower_price)
        """
        return self.engine.exposed_get_price(date)


    def start_simulation(self):
        """
        A function runs at the beginning of the simulation.
        """
        pass

    def end_simulation(self):
        """
        A function runs at the beginning of the simulation.
        """
        pass

class SimpleLeader(Leader):
    def __init__(self, name, engine):
        super().__init__(name, engine)
        self.c_L = 1

    def new_price(self, date: int):
        return 1.5

    def end_simulation(self):
        #calc profit
        total_profit = 0
        for day in range(101,131):
            sales = 100 - 5 * self.get_price_from_date(day)[0] + 3 * self.get_price_from_date(day)[1]
            profit = sales * (self.get_price_from_date(day)[0] - self.c_L)
            total_profit += profit
        self.total_profit = total_profit

group_num = 37
assert isinstance(group_num, int), f"Expected an integer for group_num, but got {type(group_num).__name__}"

class DummyLeader(Leader):
    def __init__(self, name, engine):
        super().__init__(name, engine)

In [2]:
class Leader1(Leader):
    def __init__(self, name, engine):
        super().__init__(name, engine)
        self.c_L = 1.0
        self.theta = np.array([1.0, 0.5])
        self.P = np.eye(2) * 100.0
        self.lam = 0.95
        self.price_lb = 0
        self.price_ub = None
        self.total_profit = 0

        self.u_L_hist = np.array([], dtype=float)
        self.u_F_hist = np.array([], dtype=float)
        self.hist_days = 100

    def get_ideal_price(self, theta):
        numer = 3 * (theta[1] - theta[0] - 35)
        denom = 2 * (3 * theta[1] -5)
        return numer / denom

    def new_price(self, date):
        """
        A function for setting the new price of each day.
        :param date: date of the day to be updated
        :return: (float) price for the day
        """
        #update with latest stats
        if len(self.u_L_hist) < date - 1:
            self.u_L_hist = np.append(self.u_L_hist, self.get_price_from_date(date - 1)[0])
            self.u_F_hist = np.append(self.u_F_hist, self.get_price_from_date(date - 1)[1])
        X = np.column_stack([np.ones(self.hist_days), self.u_L_hist[-self.hist_days:]])
        self.theta, _, _, _ = np.linalg.lstsq(X, self.u_F_hist[-self.hist_days:], rcond=None)

        # price bound
        ideal_price = self.get_ideal_price(self.theta)
        if ideal_price < self.price_lb:
            ideal_price = self.price_lb
        elif self.price_ub is not None and ideal_price > self.price_ub:
            ideal_price = self.price_ub

        return ideal_price

    def get_price_from_date(self, date):
        # :return: a tuple (leader_price, follower_price)
        return self.engine.exposed_get_price(date)

    def start_simulation(self):
        for day in range(1, 1+self.hist_days):
            lp, fp = self.get_price_from_date(day)
            self.u_L_hist = np.append(self.u_L_hist, lp)
            self.u_F_hist = np.append(self.u_F_hist, fp)
            # print(self.u_L_hist)
            # print(lp)

        X = np.column_stack([np.ones(self.hist_days), self.u_L_hist[-self.hist_days:]])
        self.theta, _, _, _ = np.linalg.lstsq(X, self.u_F_hist[-self.hist_days:], rcond=None)

    def end_simulation(self):
        #calc profit
        total_profit = 0
        for day in range(101,131):
            sales = 100 - 5 * self.get_price_from_date(day)[0] + 3 * self.get_price_from_date(day)[1]
            profit = sales * (self.get_price_from_date(day)[0] - self.c_L)
            total_profit += profit
        self.total_profit = total_profit


In [3]:
# ACCESS ENGINE DIRECTLY TO MASS TEST PARAMS
engine = Engine()
Leader.update_subclass_registry()
print([m for m in dir(engine) if not m.startswith('_')])

['connect', 'exposed_get_price', 'leader', 'main_loop', 'mk', 'mk_i']


In [4]:
leader_chosen = Leader1
leader_object = leader_chosen(leader_chosen.__name__, engine)
follower_chosen = 'MK1'

In [5]:
for leader in [Leader1, SimpleLeader]:
    for follower in ['MK1', 'MK2', 'MK3']:
        # RUN THIS .connect AND .main_loop TO CONNECT THEN SIMULATE
        leader_chosen = leader
        leader_object = leader_chosen(leader_chosen.__name__, engine)
        follower_chosen = follower
        print(engine.connect(leader_object, leader_chosen.__name__, follower_chosen))
        print(engine.main_loop(101,130))
        print(leader_object.total_profit)

Connected to Leader1 and MK1
[(101, np.float64(118.39186091428189), np.float64(94.34408158133016), 1.0), (102, np.float64(20.468285085503748), np.float64(15.745264417750962), 1.0), (103, np.float64(20.39554010074939), np.float64(18.333170803432818), 1.0), (104, np.float64(20.40694091492647), np.float64(17.11380597355853), 1.0), (105, np.float64(20.38181739880373), np.float64(18.324284457205493), 1.0), (106, np.float64(20.393338253986542), np.float64(16.52053719647255), 1.0), (107, np.float64(20.354486980902358), np.float64(17.827825002119287), 1.0), (108, np.float64(20.354296964439115), np.float64(15.187246907666324), 1.0), (109, np.float64(20.28377522338247), np.float64(17.889244542648658), 1.0), (110, np.float64(20.28799789792991), np.float64(16.73264706043515), 1.0), (111, np.float64(20.262879619477232), np.float64(18.76193242225002), 1.0), (112, np.float64(20.288712180241703), np.float64(18.374622538374872), 1.0), (113, np.float64(20.304150582464704), np.float64(18.41676351032822),

In [6]:
from IPython.display import Javascript

display(Javascript('''google.colab.output.setIframeHeight(0, true, {maxHeight: 5000})'''))
app = GUI(engine, Leader, group_num)

<IPython.core.display.Javascript object>